## Ensuring the Normalization process 


In [1]:
# IMPORTING THE NECESSARY LIBRARIES 

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import sqlite3 
import os 
import sys 
import path 
import csv

    




## defining the utility functions 

In [2]:
def create_connection(db_file, delete_db= False):
    if delete_db and os.path.exists(db_file):
        try:
            os.remove(db_file)
        except PermissionError:
            print(f"Error: The file '{db_file}' is currently in use by another process.")
            return None

    try:
        conn = sqlite3.connect(db_file)
        conn.execute("PRAGMA foreign_keys = 1")
        print(f"Connected to database: {db_file}")
        return conn
    except sqlite3.Error as e:
        print(f"Error connecting to database: {e}")
        return None

def create_table(conn, create_table_sql):
    try:
        c = conn.cursor()
        c.execute(create_table_sql)
        print("Table created successfully.")
    except sqlite3.Error as e:
        print(f"Error creating table: {e}")

def execute_sql_statement(sql_statement, conn, parameters=()):
    try:
        cur = conn.cursor()
        cur.execute(sql_statement, parameters)
        return cur.fetchall()
    except sqlite3.Error as e:
        print(f"Error executing SQL statement: {e}")
        return None
       
def insert_data(conn, query, data):
    try:
        cur = conn.cursor()
        cur.executemany(query, data)
        conn.commit()
        print("Data inserted successfully.")
    except sqlite3.Error as e:
        print(f"Error inserting data: {e}")
       

def fetch_data_from_db(conn, query):
    try:
        df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()



def load_data_from_csv(csv_file, conn):
    with open(csv_file, 'r') as file:
        reader = csv.reader(file)
        headers = next(reader)  # Skip the header row

        # Normalized data lists
        applicants_data = []
        financial_data = []
        credit_history_data = []
        employment_data = []
        loan_data = []

        applicant_id = 1  # Manual primary key tracking
       
        for row in reader:
            if '?' in row or '' in row:
                continue  # Skip rows with missing data

            # Normalize data for each table
            applicants_data.append((row[0], int(row[1]), row[9], int(row[10]), row[11]))  # ApplicationDate, Age, MaritalStatus, NumberOfDependents, HomeOwnershipStatus
            financial_data.append((applicant_id, int(row[2]), float(row[26]), int(row[22]), int(row[23]),
                                   int(row[24]), int(row[25]), int(row[29]), float(row[16]), float(row[33])))
            credit_history_data.append((applicant_id, int(row[3]), int(row[20]), int(row[14]), int(row[15]),
                                        float(row[13]), int(row[17]), int(row[19]), float(row[27]), int(row[21])))
            employment_data.append((applicant_id, row[4], row[5], int(row[6]), int(row[28])))  # EmploymentStatus, EducationLevel, Experience, JobTenure
            loan_data.append((applicant_id, int(row[7]), int(row[8]), row[18], float(row[32]), float(row[30]),
                              float(row[31]), int(row[34]), float(row[35])))

            applicant_id += 1

    return applicants_data, financial_data, credit_history_data, employment_data, loan_data


In [6]:




create_applicants_table = """
CREATE TABLE Applicants (
ApplicantID INTEGER PRIMARY KEY AUTOINCREMENT,
ApplicationDate DATE NOT NULL,
Age INT NOT NULL,
MaritalStatus VARCHAR(20),
NumberOfDependents INT,
HomeOwnershipStatus VARCHAR(50)
);
"""

create_financial_details_table = """
CREATE TABLE FinancialDetails (
FinancialID INTEGER PRIMARY KEY AUTOINCREMENT,
ApplicantID INT NOT NULL,
AnnualIncome INT,
MonthlyIncome FLOAT,
SavingsAccountBalance INT,
CheckingAccountBalance INT,
TotalAssets INT,
TotalLiabilities INT,
NetWorth INT,
DebtToIncomeRatio FLOAT,
TotalDebtToIncomeRatio FLOAT,
FOREIGN KEY (ApplicantID) REFERENCES Applicants(ApplicantID)
);
"""

create_credit_history_table = """
CREATE TABLE CreditHistory (
CreditHistoryID INTEGER PRIMARY KEY AUTOINCREMENT,
ApplicantID INT NOT NULL,
CreditScore INT,
PaymentHistory INT,
NumberOfOpenCreditLines INT,
NumberOfCreditInquiries INT,
CreditCardUtilizationRate FLOAT,
BankruptcyHistory INT,
PreviousLoanDefaults INT,
UtilityBillsPaymentHistory FLOAT,
LengthOfCreditHistory INT,
FOREIGN KEY (ApplicantID) REFERENCES Applicants(ApplicantID)
);
"""

create_employment_details_table = """
CREATE TABLE EmploymentDetails (
EmploymentID INTEGER PRIMARY KEY AUTOINCREMENT,
ApplicantID INT NOT NULL,
EmploymentStatus VARCHAR(50),
EducationLevel VARCHAR(50),
Experience INT,
JobTenure INT,
FOREIGN KEY (ApplicantID) REFERENCES Applicants(ApplicantID)
);
"""

create_loan_details_table = """
CREATE TABLE LoanDetails (
LoanID INTEGER PRIMARY KEY AUTOINCREMENT,
ApplicantID INT NOT NULL,
LoanAmount INT,
LoanDuration INT,
LoanPurpose VARCHAR(100),
MonthlyLoanPayment FLOAT,
BaseInterestRate FLOAT,
InterestRate FLOAT,
LoanApproved BOOLEAN,
RiskScore FLOAT,
FOREIGN KEY (ApplicantID) REFERENCES Applicants(ApplicantID)
);
"""

join_query = """
    SELECT
        A.ApplicantID, A.ApplicationDate, A.Age, A.MaritalStatus, A.NumberOfDependents, A.HomeOwnershipStatus,
        F.AnnualIncome, F.MonthlyIncome, F.SavingsAccountBalance, F.CheckingAccountBalance,
        F.TotalAssets, F.TotalLiabilities, F.NetWorth, F.DebtToIncomeRatio, F.TotalDebtToIncomeRatio,
        C.CreditScore, C.PaymentHistory, C.NumberOfOpenCreditLines, C.NumberOfCreditInquiries,
        C.CreditCardUtilizationRate, C.BankruptcyHistory, C.PreviousLoanDefaults, C.UtilityBillsPaymentHistory,
        C.LengthOfCreditHistory,
        E.EmploymentStatus, E.EducationLevel, E.Experience, E.JobTenure,
        L.LoanAmount, L.LoanDuration, L.LoanPurpose, L.MonthlyLoanPayment, L.BaseInterestRate,
        L.InterestRate, L.LoanApproved, L.RiskScore
    FROM Applicants A
    LEFT JOIN FinancialDetails F ON A.ApplicantID = F.ApplicantID
    LEFT JOIN CreditHistory C ON A.ApplicantID = C.ApplicantID
    LEFT JOIN EmploymentDetails E ON A.ApplicantID = E.ApplicantID
    LEFT JOIN LoanDetails L ON A.ApplicantID = L.ApplicantID;
    """



def main():
    db_file = "LoanDATABASE.db"
    csv_file = '/Users/ajit/Desktop/loan_predictor_app/Loan.csv'
    conn = create_connection(db_file, delete_db=True)
    if conn is None:
        return

    create_table(conn, create_applicants_table)
    create_table(conn, create_financial_details_table)
    create_table(conn, create_credit_history_table)
    create_table(conn, create_employment_details_table)
    create_table(conn, create_loan_details_table)

    applicants_data, financial_data, credit_history_data, employment_data, loan_data = load_data_from_csv(csv_file, conn)


    insert_data(conn, """
    INSERT INTO Applicants (ApplicationDate, Age, MaritalStatus, NumberOfDependents, HomeOwnershipStatus)
    VALUES (?, ?, ?, ?, ?);
    """, applicants_data)

    insert_data(conn, """
    INSERT INTO FinancialDetails (ApplicantID, AnnualIncome, MonthlyIncome, SavingsAccountBalance, CheckingAccountBalance,
    TotalAssets, TotalLiabilities, NetWorth, DebtToIncomeRatio, TotalDebtToIncomeRatio)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, financial_data)

    insert_data(conn, """
    INSERT INTO CreditHistory (ApplicantID, CreditScore, PaymentHistory, NumberOfOpenCreditLines, NumberOfCreditInquiries,
    CreditCardUtilizationRate, BankruptcyHistory, PreviousLoanDefaults, UtilityBillsPaymentHistory,
    LengthOfCreditHistory)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, credit_history_data)

    insert_data(conn, """
    INSERT INTO EmploymentDetails (ApplicantID, EmploymentStatus, EducationLevel, Experience, JobTenure)
    VALUES (?, ?, ?, ?, ?);
    """, employment_data)

    insert_data(conn, """
    INSERT INTO LoanDetails (ApplicantID, LoanAmount, LoanDuration, LoanPurpose, MonthlyLoanPayment, BaseInterestRate,
    InterestRate, LoanApproved, RiskScore)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, loan_data)

    conn = sqlite3.connect('LoanDATABASE.db')

    df = fetch_data_from_db(conn,join_query)
    print(df)



    conn.commit()
    conn.close()

    print("Tables created, data inserted, and connection closed.")

    return df

